# Clase 2.2 - pytest Avanzado: Fixtures, Parametrizacion y Mocks

**Unidad:** 2 - Tecnicas de Desarrollo
**Duracion:** 2 horas
**Autor:** IF0100 - UNAULA

## Objetivos de Aprendizaje

Al finalizar esta clase, sera capaz de:
- [ ] Crear y usar fixtures de pytest para setup compartido
- [ ] Aplicar scopes de fixtures (function, class, module, session)
- [ ] Usar parametrizacion para ejecutar tests con multiples datos
- [ ] Crear mocks y patches con unittest.mock
- [ ] Aplicar monkeypatch para modificar comportamiento en tests
- [ ] Organizar tests en clases y usar fixtures de conftest.py

---

## 1. Conceptos Teoricos

### Por que pytest Avanzado?

En la clase anterior aprendimos lo basico de pytest: escribir tests con `assert`.
Ahora vamos a explorar caracteristicas avanzadas que hacen que nuestros tests sean:

- **Mas mantenibles:** Fixtures eliminan duplicacion
- **Mas completos:** Parametrizacion prueba mas casos
- **Mas aislados:** Mocks aislan dependencias externas

### Fixtures: El Poder de pytest

Una **fixture** es una funcion que prepara el entorno para un test.
Puede crear datos, inicializar recursos, configurar estado, etc.

**Analogia:** Una fixture es como el "setup" de un laboratorio antes de un experimento.

**Ventajas de Usar Fixtures:**
- **Reutilizacion:** Escribir una vez, usar en muchos tests
- **Legibilidad:** Los tests son mas claros y enfocados
- **Mantenibilidad:** Cambios en un lugar afectan todos los tests
- **Inyeccion de dependencias:** pytest las inyecta automaticamente

---

## 2. Fixtures en pytest

### Sintaxis Basica

```python
import pytest

@pytest.fixture
def usuario():
    """Fixture que crea un usuario de prueba."""
    return Usuario(username="testuser", email="test@example.com")

def test_username(usuario):  # pytest inyecta la fixture automaticamente
    """Test que el username es correcto."""
    assert usuario.username == "testuser"
    assert usuario.email == "test@example.com"
```

**Ejemplo practico:**

In [ ]:
# Ejemplo 1: Fixture basica
import pytest

# Definimos una clase simple para el ejemplo
class Usuario:
    def __init__(self, username, email):
        self.username = username
        self.email = email

@pytest.fixture
def usuario_test():
    """Fixture que crea un usuario de prueba."""
    return Usuario(username="testuser", email="test@example.com")

# Test que usa la fixture
def test_username(usuario_test):
    """Test que el username es correcto."""
    assert usuario_test.username == "testuser"
    assert usuario_test.email == "test@example.com"

# Simular ejecucion del test
print("Ejecutando test_username...")
usuario = Usuario(username="testuser", email="test@example.com")
print(f"Username: {usuario.username}")
print(f"Email: {usuario.email}")
print("Test PASSED!")

### Fixture con Setup y Teardown

El flujo de una fixture con `yield` es:

1. **SETUP:** Codigo antes del `yield`
2. **TEST:** Se ejecuta el codigo del test
3. **TEARDOWN:** Codigo despues del `yield`

**Ejemplo:**

In [ ]:
# Ejemplo 2: Fixture con setup y teardown
import os
import tempfile

@pytest.fixture
def archivo_temporal():
    """Fixture que crea un archivo temporal y lo limpia despues."""
    # SETUP: Crear archivo temporal
    fd, ruta = tempfile.mkstemp()
    with os.fdopen(fd, 'w') as f:
        f.write("contenido de prueba")
    
    # Yield el objeto al test
    yield ruta
    
    # TEARDOWN: Limpiar archivo
    os.unlink(ruta)

# Simulacion del test
def test_leer_archivo_temporal():
    """Test que lee el archivo temporal creado por la fixture."""
    # Crear archivo temporal manualmente para demostracion
    fd, ruta = tempfile.mkstemp()
    with os.fdopen(fd, 'w') as f:
        f.write("contenido de prueba")
    
    # Test: leer contenido
    with open(ruta, 'r') as f:
        contenido = f.read()
    
    assert contenido == "contenido de prueba"
    print(f"Contenido leido: {contenido}")
    
    # Cleanup manual
    os.unlink(ruta)
    print("Test PASSED! Archivo temporal eliminado.")

test_leer_archivo_temporal()

### Scopes de Fixtures

El scope define cuantas veces se crea la fixture durante la ejecucion de tests.

| Scope | Frecuencia | Uso Tipico |
|-------|-----------|------------|
| `function` (default) | Una vez por test | Datos especificos del test |
| `class` | Una vez por clase de tests | Setup compartido por clase |
| `module` | Una vez por modulo | Recursos costosos (conexion BD) |
| `session` | Una vez por sesion de pytest | Recursos globales (configuracion) |

**Ejemplo de scope module:**

In [ ]:
# Ejemplo 3: Fixture con scope module
@pytest.fixture(scope="module")
def conexion_db():
    """Fixture de modulo: una conexion compartida para todos los tests."""
    print("\n[SETUP] Creando conexion a BD...")
    conexion = {"conexion": "simulada", "datos": []}
    yield conexion
    print("\n[TEARDOWN] Cerrando conexion a BD...")

# Simulacion de tests compartiendo estado
conexion = {"conexion": "simulada", "datos": []}

def test_agregar_dato():
    """Test que agrega un dato a la BD."""
    conexion["datos"].append("dato1")
    assert len(conexion["datos"]) == 1
    print(f"Datos despues de test 1: {conexion['datos']}")

def test_agregar_otro_dato():
    """Test que agrega otro dato (conserva estado)."""
    # dato1 sigue ahi porque es la misma conexion
    conexion["datos"].append("dato2")
    assert len(conexion["datos"]) == 2
    print(f"Datos despues de test 2: {conexion['datos']}")

# Ejecutar tests
print("=== Test 1 ===")
test_agregar_dato()
print("\n=== Test 2 ===")
test_agregar_otro_dato()
print("\nAmbos tests comparten la misma conexion!")

---

## 3. Parametrizacion

### Basico: @pytest.mark.parametrize

La parametrizacion permite ejecutar el mismo test con diferentes datos:

In [ ]:
# Ejemplo 4: Test parametrizado basico
import re

def validar_email(email):
    """Valida si un email tiene formato correcto."""
    patron = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(patron, email))

# Casos de prueba parametrizados
casos_email = [
    ("usuario@example.com", True),     # Email valido
    ("user.name@example.com", True),   # Email con punto
    ("user+tag@example.com", True),    # Email con +
    ("invalido", False),               # Sin @
    ("@example.com", False),           # Sin usuario
    ("usuario@", False),               # Sin dominio
]

print("Ejecutando tests parametrizados de validacion de email:\n")
for email, esperado in casos_email:
    resultado = validar_email(email)
    estado = "PASSED" if resultado == esperado else "FAILED"
    print(f"test_validar_email[{email}] - {estado}")
    assert resultado == esperado, f"Fallo para {email}"

print("\n¡Todos los tests pasaron!")

---

## 4. Mocks y Monkeypatch

### Que es un Mock?

Un **mock** es un objeto simulado que imita el comportamiento de un objeto real.
Nos permite:

- Aislar el codigo bajo prueba
- Controlar respuestas de dependencias externas
- Verificar interacciones (llamadas a metodos)
- Simular errores y casos excepcionales

### unittest.mock.Mock Basico

In [ ]:
# Ejemplo 5: Mock basico
from unittest.mock import Mock

# Crear un mock simple
mock_db = Mock()

# Configurar retorno
class UsuarioMock:
    def __init__(self, id, username):
        self.id = id
        self.username = username

mock_db.obtener_usuario.return_value = UsuarioMock(id=1, username="test")

# Usar el mock
usuario = mock_db.obtener_usuario(1)
print(f"Usuario obtenido: {usuario.username}")
assert usuario.username == "test"

# Verificar que se llamo
print(f"Fue llamado: {mock_db.obtener_usuario.called}")
print(f"Veces llamado: {mock_db.obtener_usuario.call_count}")

print("\nMock basico - PASSED!")

### Verificar Interacciones

Los mocks permiten verificar que se llamaron correctamente:

In [ ]:
# Ejemplo 6: Verificar interacciones con mocks
from unittest.mock import Mock

class ProyectoService:
    def __init__(self, repo):
        self.repo = repo
    
    def crear_proyecto(self, nombre, usuario_id):
        proyecto = Mock(nombre=nombre, usuario_id=usuario_id)
        return self.repo.guardar(proyecto)

# Crear mock del repositorio
mock_repo = Mock()
mock_repo.guardar.return_value = Mock(id=1, nombre="Nuevo Proyecto", usuario_id=1)

# Crear servicio con el mock
service = ProyectoService(mock_repo)

# Ejecutar accion
resultado = service.crear_proyecto("Nuevo Proyecto", usuario_id=1)

# Verificar que se llamo al repo
print(f"Fue llamado guardar: {mock_repo.guardar.called}")
print(f"Veces llamado: {mock_repo.guardar.call_count}")
print(f"Argumentos de la llamada: {mock_repo.guardar.call_args}")

# Verificar numero de llamadas
assert mock_repo.guardar.call_count == 1

print("\nVerificacion de interacciones - PASSED!")

---

## 5. Comandos Utiles de pytest

| Comando | Descripcion |
|----------|------------|
| `pytest --fixtures` | Listar todas las fixtures disponibles |
| `pytest -v --tb=short` | Output verbose con traceback corto |
| `pytest -k "test_crear"` | Ejecutar tests que contengan "test_crear" |
| `pytest -x` | Detenerse en el primer fallo |
| `pytest --lf` | Ejecutar solo los tests que fallaron la ultima vez |
| `pytest tests/test_api/` | Ejecutar solo tests de un directorio |
| `pytest --cov=src` | Ejecutar tests con coverage |

---

## Resumen

### Conceptos Clave

| Concepto | Sintaxis | Uso |
|----------|----------|-----|
| Fixture | `@pytest.fixture` | Setup compartido |
| Scope | `scope="module"` | Frecuencia de creacion |
| Parametrize | `@pytest.mark.parametrize` | Multiples casos de prueba |
| Mock | `Mock()` | Objeto simulado |
| Monkeypatch | `monkeypatch.setattr` | Modificar comportamiento |
| TestClient | `TestClient(app)` | Testing de APIs FastAPI |
| conftest.py | `tests/conftest.py` | Fixtures compartidas |

### Checklist de Aprendizaje

- [ ] Entiendo que son las fixtures y cuando usarlas
- [ ] Se usar diferentes scopes de fixtures
- [ ] Puedo crear tests parametrizados
- [ ] Se usar mocks para aislar dependencias
- [ ] Conozco monkeypatch para modificar comportamiento
- [ ] Puedo organizar fixtures en conftest.py

### Para Profundizar

- [pytest Fixtures - Documentacion Oficial](https://docs.pytest.org/en/stable/explanation/fixtures.html)
- [pytest Parametrizacion - Documentacion Oficial](https://docs.pytest.org/en/stable/parametrize.html)
- [unittest.mock - Documentacion Python](https://docs.python.org/3/library/unittest.mock.html)
- [Real Python: A Practical Guide to pytest](https://realpython.com/pytest-python-testing/)

---

**¡Siguiente clase:** BDD con behave

**Tarea para casa:** Crear fixtures compartidas para el proyecto TaskFlow